<a href="https://colab.research.google.com/github/hyui02/ESAA_assignment/blob/main/%08ESAA_WEEK13_%EC%88%98%EC%83%81%EC%9E%91%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 수상작 리뷰: NASA Software Defect Prediction (SDP) Challenge

- 리뷰 노트북: [NASA SDP - EDA | Baseline](https://www.kaggle.com/code/hadriencr/nasa-sdp-eda-baseline) (Hadrien CR.)
- 대회: [NASA Software Defect Prediction (SDP) Challenge (Kaggle, 2024)](https://www.kaggle.com/competitions/nasa-software-defect-prediction-sdp-challenge)
- 작성자 Hadrien CR.은 이 대회의 최종 Private 리더보드 **1위(0.88784)** 이고, 이 노트북은 그가 공개한 EDA와 기본(Baseline) 모델이다. 리뷰한 버전(Version 24)의 Private 점수는 0.86421이다.
- 한 줄 요약: 소프트웨어 모듈의 코드 지표 21개로 결함 여부를 예측하는 불균형 이진 분류 문제를, **로그 변환 → ADASYN 오버샘플링 → 랜덤 포레스트** 순서의 짧은 파이프라인으로 풀었다.

---

# **1. 데이터 설명**

## 1-1. 대회 과제
- NASA 소프트웨어의 각 모듈에 대해 측정한 코드 지표로 **결함이 있는 모듈(Defect=1)과 없는 모듈(Defect=0)을 구분**하는 이진 분류 문제이다.
- 개발 초기에 결함 가능성이 높은 모듈을 찾아내면, 배포 전에 문제를 고쳐 임무 실패를 막고 수정 비용도 줄일 수 있다는 것이 대회의 배경이다.
- 평가지표는 **ROC-AUC**(ROC 곡선 아래 면적)이다. 0/1을 맞히는 정확도가 아니라 **결함일 확률**을 얼마나 잘 순서대로 매기는지를 평가하므로, 제출할 때도 `predict()`가 아닌 `predict_proba()`로 구한 확률을 낸다.
- 제출 파일은 `Index`, `Defect` 두 열로 구성한다.
- 2024년 3월 10일 ~ 4월 15일에 진행된 커뮤니티 대회로, 19개 팀이 참가하였다.

## 1-2. 파일 구성
| 파일 | 내용 |
|---|---|
| `Train_Features.csv` | 학습 데이터의 피처. 1,676개 모듈 |
| `Train_Output.csv` | 학습 데이터의 타깃(`Defect`) |
| `Test_Features.csv` | 테스트 데이터의 피처. 419개 모듈 |
| `Sample_Submission.csv` | 제출 파일 형식 예시 |

- 피처와 타깃이 서로 다른 파일로 나뉘어 있어서, 노트북은 `Train_Output.csv`의 `Defect` 열을 `Train_Features.csv`에 붙여서 사용한다.

## 1-3. 컬럼 구성
`Index`를 제외한 피처 21개는 모두 숫자형(정수 또는 실수)이고, 성격에 따라 네 묶음으로 나눌 수 있다.

| 묶음 | 컬럼 | 의미 |
|---|---|---|
| 코드 줄 수(LOC) | `LOC_BLANK`, `LOC_CODE_AND_COMMENT`, `LOC_COMMENTS`, `LOC_EXECUTABLE`, `LOC_TOTAL` | 빈 줄, 주석, 실행 코드 등 줄 수 |
| 복잡도 | `BRANCH_COUNT`, `CYCLOMATIC_COMPLEXITY`, `DESIGN_COMPLEXITY`, `ESSENTIAL_COMPLEXITY` | 분기 수, 순환 복잡도 등 코드 흐름의 복잡한 정도 |
| Halstead 지표 | `HALSTEAD_CONTENT`, `HALSTEAD_DIFFICULTY`, `HALSTEAD_EFFORT`, `HALSTEAD_ERROR_EST`, `HALSTEAD_LENGTH`, `HALSTEAD_LEVEL`, `HALSTEAD_PROG_TIME`, `HALSTEAD_VOLUME` | 연산자·피연산자 수로 계산한 코드의 크기, 난이도, 예상 작업량 |
| 연산자·피연산자 수 | `NUM_OPERANDS`, `NUM_OPERATORS`, `NUM_UNIQUE_OPERANDS`, `NUM_UNIQUE_OPERATORS` | Halstead 지표를 계산하는 기본 재료 |
| **타깃** | `Defect` | 결함 여부 (1: 결함 있음, 0: 없음) |

## 1-4. 데이터를 다룰 때 알아 둘 특징
- **결측치가 없다**: 모든 컬럼의 결측치가 0개라 따로 채울 필요가 없다.
- **클래스 불균형**: 결함 모듈(Defect=1)의 비율이 약 15.5%로, 결함이 없는 모듈이 훨씬 많다. 그래서 정확도로 평가하면 모두 0이라고 찍어도 84% 이상이 나오고, 이 때문에 대회도 ROC-AUC로 평가한다.
- **한쪽으로 치우친 분포**: 줄 수나 Halstead 지표는 대부분의 모듈에서 작고, 일부 큰 모듈에서만 매우 큰 값을 가진다. 노트북이 모든 피처에 로그 변환을 적용하는 이유이다.
- **피처끼리 상관이 높다**: Halstead 지표는 연산자·피연산자 수로부터 계산되고, 줄 수와 복잡도도 코드 크기가 커지면 함께 커지기 때문에 서로 비슷하게 움직이는 피처가 많다.

---

# **2. 코드 흐름**

1. **데이터 불러오기**
    - 피처 파일과 타깃 파일을 읽고, 학습 피처에 `Defect` 열을 붙인다.
2. **결측치·자료형 확인과 EDA**
    - 결측치가 없는 것을 확인하고, 피처별 히스토그램을 결함 여부로 나눠 그린다.
    - 상관관계 히트맵으로 피처 간 관계를 확인한다.
3. **피처 변환**
    - 모든 피처에 `log(x+1)` 변환을 적용한다.
    - `StandardScaler`로 표준화를 시도한다(다만 결과를 저장하지 않아 실제로는 적용되지 않는다. 3-③에서 다룬다).
4. **검증 세트 분리와 오버샘플링**
    - 결함 비율을 유지하도록 `stratify=y`로 학습 90%, 검증 10%를 나눈다.
    - 학습 데이터에만 **ADASYN**을 적용해 두 클래스를 각각 1.5배로 늘린다.
5. **모델 학습과 평가**
    - `RandomForestClassifier(n_estimators=1000, max_depth=40)`로 학습하고, 검증 세트의 ROC-AUC를 확인한다(노트북 기준 약 0.754).
6. **제출**
    - 테스트 데이터의 결함 확률을 `predict_proba()`로 구해 `submission.csv`로 저장한다.

---

# **3. 주요 코드 분석**

- 코랩에 업로드한 대회 데이터(`/content/` 경로)를 그대로 사용해 원본 노트북의 흐름을 따라간다. 파일 이름이 다르면 첫 코드 셀의 경로만 바꾸면 된다.
- 원본은 `train_test_split`과 ADASYN에 무작위 시드를 고정하지 않아 실행할 때마다 결과가 조금씩 다르다. 여기서는 결과를 다시 확인할 수 있도록 `random_state=0`을 넣었다.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

try:
    from imblearn.over_sampling import ADASYN
except ImportError:
    !pip -q install imbalanced-learn
    from imblearn.over_sampling import ADASYN

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

DATA_DIR = '/content/'

**① 데이터 불러오기와 기본 확인**

In [ ]:
train_df = pd.read_csv(DATA_DIR + 'Train_Features.csv')
train_output_df = pd.read_csv(DATA_DIR + 'Train_Output.csv')
test_df = pd.read_csv(DATA_DIR + 'Test_Features.csv')
train_df['Defect'] = train_output_df['Defect']

print('학습 데이터:', train_df.shape, '/ 테스트 데이터:', test_df.shape)
print('결측치 개수 합계:', train_df.isnull().sum().sum())
print('결함 비율:', round(train_df['Defect'].mean(), 4))
train_df.head(3)

- 피처 파일과 타깃 파일이 따로 있어서, 같은 순서로 저장되어 있다는 전제 아래 `Defect` 열을 그대로 옆에 붙였다. 두 파일 모두 `Index` 열이 있으므로 `merge`로 붙이면 순서가 다른 경우에도 안전하다.
- 결측치가 전혀 없고 모든 컬럼이 숫자형이라 인코딩이나 결측치 처리 없이 바로 분석할 수 있다.
- 결함 비율이 약 15.5%라서, 이후 오버샘플링과 평가지표 선택이 이 불균형을 중심으로 이루어진다.

**② EDA: 결함 여부별 분포와 상관관계**

In [ ]:
plt.figure(figsize=(15, 25))
feature_cols = [c for c in train_df.columns if c not in ['Index', 'Defect']]

plt.subplot(8, 3, 1)
sns.histplot(data=train_df, x='Defect')
plt.title('Defect')
for i, column in enumerate(feature_cols):
    plt.subplot(8, 3, i + 2)
    sns.histplot(data=train_df[0:300], x=column, hue='Defect', multiple='stack', log_scale=(False, True))
    plt.title(column)
    plt.xlabel('')
    plt.ylabel('')
    plt.xticks(rotation=70)
plt.tight_layout()
plt.show()

- 피처마다 히스토그램을 그리고 `hue='Defect'`로 결함 여부에 따라 색을 나눠 쌓았다(`multiple='stack'`). 같은 구간 안에서 결함 모듈의 비중이 어떻게 달라지는지 볼 수 있다.
- `log_scale=(False, True)`로 **y축(개수)만 로그**를 씌웠다. 값이 작은 구간에 모듈이 몰려 있어서, 그대로 그리면 큰 값 구간의 막대가 거의 보이지 않기 때문이다.
- 이 그래프로 줄 수나 복잡도가 큰 구간에서 결함 모듈의 비중이 커지는지, 즉 "코드가 길고 복잡할수록 결함이 생기기 쉽다"는 직관이 데이터에서도 보이는지 확인할 수 있다. 아래 상관계수 출력과 함께 보면 어떤 피처가 결함과 관련이 큰지 알 수 있다.
- 원본은 시각화를 빠르게 하려고 앞의 300개 행만 사용했다(`train_df[0:300]`). 전체 분포를 보려면 이 부분을 빼면 된다.

In [ ]:
plt.figure(figsize=(16, 13))
sns.heatmap(train_df.drop(columns=['Index']).corr(), annot=True, cmap='coolwarm', fmt='.2f', annot_kws={'size': 7})
plt.title('Correlation Heatmap')
plt.show()

corr_with_target = train_df.drop(columns=['Index']).corr()['Defect'].drop('Defect').sort_values(ascending=False)
print('Defect와의 상관계수 상위 5개:\n', corr_with_target.head(5).round(3))

- 피처끼리 상관계수가 높은 칸(붉은색)이 넓게 나타난다. Halstead 지표들은 연산자·피연산자 수로 계산되는 값이라 서로 강하게 연결되어 있고, 줄 수와 복잡도도 코드 크기와 함께 커진다.
- 선형 모델이었다면 이런 다중공선성이 회귀 계수를 불안정하게 만들었겠지만, 이 노트북이 사용하는 **랜덤 포레스트는 트리 기반이라 피처 간 상관에 덜 민감**해서 피처를 줄이지 않고 그대로 사용했다.
- 원본은 `Index` 열까지 포함해 상관계수를 구했는데, `Index`는 단순한 번호라 분석에 의미가 없으므로 위 코드에서는 제외했다.

**③ 로그 변환과 표준화**

In [ ]:
y = train_df['Defect']
X = train_df.drop(columns=['Defect', 'Index'])
X_eval = test_df.drop(columns=['Index'])

# 원본: math.log(x+1)을 apply로 한 칸씩 적용 -> np.log1p로 같은 계산을 한 번에
X = np.log1p(X)
X_eval = np.log1p(X_eval)

# 원본 코드 그대로: 결과를 변수에 저장하지 않음
scaler = StandardScaler()
scaler.fit_transform(X)
scaler.transform(X_eval)

print('표준화 후 X의 평균(앞 3개 피처):', X.mean().round(3).values[:3])
print('표준화 후 X의 표준편차(앞 3개 피처):', X.std().round(3).values[:3])

- 모든 피처에 `log(x+1)`을 적용해 큰 값 쪽으로 길게 늘어진 분포를 좁혔다. 0인 값이 많기 때문에 `log(x)`가 아니라 `log(x+1)`을 써야 한다. 원본은 `math.log`를 `apply`로 한 칸씩 계산했는데, `np.log1p`를 쓰면 같은 결과를 한 번에 얻을 수 있다.
- **원본의 실수**: `scaler.fit_transform(X)`의 결과를 `X`에 다시 저장하지 않아서, 표준화가 실제로는 적용되지 않았다. 위 출력에서 평균이 0, 표준편차가 1이 아닌 것으로 확인할 수 있다. 제대로 적용하려면 `X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)`처럼 결과를 저장해야 한다.
- 다만 이 노트북의 모델은 랜덤 포레스트라서, 결과에는 거의 영향이 없다. 트리는 피처 값의 **순서**로 분기 기준을 정하기 때문에 표준화처럼 값의 크기만 바꾸는 변환에는 결과가 달라지지 않는다. 반면 ADASYN은 거리를 기준으로 새 데이터를 만들기 때문에, 표준화를 제대로 했다면 오버샘플링 결과는 조금 달라졌을 수 있다.

**④ 검증 세트 분리와 ADASYN 오버샘플링**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.1, random_state=0)

ratio = 1.5
count_0, count_1 = y_train.value_counts()
target = {0: int(ratio * count_0), 1: int(ratio * count_1)}

adasyn = ADASYN(sampling_strategy=target, random_state=0)
X_train_over, y_train_over = adasyn.fit_resample(X_train, y_train)

print('오버샘플링 전:', y_train.value_counts().to_dict())
print('목표 개수   :', target)
print('오버샘플링 후:', y_train_over.value_counts().to_dict())
print('결함 비율 전/후:', round(y_train.mean(), 3), '/', round(y_train_over.mean(), 3))

- `stratify=y`로 학습·검증 세트 모두 결함 비율이 약 15.5%로 유지되도록 나눴다. 불균형 데이터에서 검증 세트에 결함 모듈이 너무 적게 들어가는 것을 막기 위해서이다.
- **ADASYN**은 SMOTE처럼 기존 데이터 사이에 새로운 데이터를 만들어 넣는 오버샘플링 방법인데, 주변에 다른 클래스가 많아 **분류하기 어려운 데이터 주위에 더 많이** 만든다는 점이 다르다.
- `sampling_strategy`에 클래스별 목표 개수를 넣어 **두 클래스를 모두 1.5배**로 늘렸다. 일반적으로 오버샘플링은 소수 클래스만 늘려 비율을 맞추는데, 이 노트북은 비율은 그대로 두고 전체 데이터 양을 늘렸다는 점이 특이하다. 그래서 오버샘플링 후에도 결함 비율이 원래와 비슷하다. 작성자는 이렇게 해서 모델이 예측한 결함 확률의 평균이 실제 결함 비율과 비슷하게 나오도록 했다(⑥에서 확인).
- ADASYN은 데이터마다 만들 개수를 주변 분포에 따라 나눠 정하기 때문에, 오버샘플링 후 개수가 목표 개수와 정확히 같지 않고 조금 적게 나올 수 있다.
- 원본은 `ADASYN(..., n_jobs=-1)`을 썼는데, 최신 imbalanced-learn에서는 `n_jobs` 인자가 없어져 에러가 날 수 있어 빼고 `random_state`를 넣었다.
- 오버샘플링은 **학습 데이터에만** 적용하고 검증 데이터에는 적용하지 않았다. 검증 데이터까지 늘리면 만들어 낸 데이터로 평가하게 되어 점수가 실제보다 좋게 나오기 때문에, 순서를 올바르게 지킨 부분이다.

**⑤ 랜덤 포레스트 학습과 ROC-AUC 평가**

In [ ]:
rfc = RandomForestClassifier(n_estimators=1000, max_depth=40, random_state=0, n_jobs=-1)
rfc.fit(X_train_over, y_train_over)
y_pred = rfc.predict_proba(X_test)[:, 1]

print('검증 ROC-AUC:', round(roc_auc_score(y_test, y_pred), 4))
print('예측 확률 평균:', round(np.mean(y_pred), 4), '/ 실제 결함 비율:', round(np.mean(y_test), 4))

- 트리 1,000개, 최대 깊이 40으로 꽤 크고 깊은 랜덤 포레스트를 사용했다. 데이터가 1,600여 개로 작아서 이 정도로 크게 만들어도 학습이 빠르다.
- 평가는 `predict()`가 아니라 `predict_proba()[:, 1]`로 **결함일 확률**을 구해 ROC-AUC를 계산했다. 원본 노트북의 검증 ROC-AUC는 약 **0.754**였다. 검증 데이터가 168개(전체의 10%)로 작아서, 나누는 방식에 따라 점수가 크게 달라질 수 있다.
- 작성자는 예측 확률의 평균(0.152)과 실제 결함 비율(0.155)을 함께 출력해서, 모델이 결함 클래스를 지나치게 적게 또는 많게 예측하지 않는지 확인했다.

**⑥ 테스트 데이터 예측과 제출 파일 생성**

In [ ]:
predictions = rfc.predict_proba(X_eval)[:, 1]
print('테스트 예측 개수:', predictions.shape, '/ 예측 확률 평균:', round(np.mean(predictions), 4), '/ 학습 데이터 결함 비율:', round(np.mean(y), 4))

predictions_df = pd.DataFrame({'Index': test_df['Index'], 'Defect': predictions})
predictions_df.to_csv('submission.csv', index=False)
predictions_df.head()

- 테스트 데이터 419개의 결함 확률을 구해 `Index`, `Defect` 형식으로 저장했다. 원본은 DataFrame의 행 번호를 `Index`로 썼는데, 위 코드는 테스트 파일의 `Index` 열을 그대로 가져와 순서가 어긋날 위험을 없앴다.
- 원본에는 `#predictions*=np.mean(y)/np.mean(predictions)`라는 코드가 주석 처리되어 있고, 옆에 "very bad idea"라고 적혀 있다. 예측 확률 평균을 실제 결함 비율에 맞추려고 모든 확률에 같은 값을 곱하는 방법인데, **ROC-AUC는 예측값의 순서만 보기 때문에** 모든 값에 같은 수를 곱해도 점수가 변하지 않는다. 작성자가 시도해 보고 의미가 없다는 것을 확인한 흔적으로 보인다.

---

# **4. 새로 알게 된 점 / 배운 점**

**배운 점**
- **불균형 데이터는 평가지표부터 다르게 본다**: 결함 비율이 15.5%라 정확도로는 모델을 제대로 평가할 수 없고, 확률의 순서를 평가하는 ROC-AUC를 써야 한다. 제출할 때도 0/1이 아니라 `predict_proba()`로 구한 확률을 낸다는 점을 다시 확인했다.
- **ADASYN 오버샘플링**: SMOTE와 비슷하지만 분류하기 어려운 데이터 주변에 더 많은 데이터를 만든다는 차이가 있다. 또 `sampling_strategy`에 클래스별 목표 개수를 직접 넣어 소수 클래스뿐 아니라 두 클래스를 모두 늘릴 수 있다는 것을 알게 되었다.
- **오버샘플링은 데이터를 나눈 다음 학습 데이터에만**: 검증 데이터까지 늘리면 만들어 낸 데이터로 평가하게 되어 점수가 실제보다 좋게 나온다. 이 노트북은 순서를 올바르게 지켰다.
- **트리 모델과 스케일링**: 랜덤 포레스트는 값의 순서로 분기하기 때문에 표준화 여부가 결과에 거의 영향을 주지 않는다. 그래서 원본에서 표준화가 실수로 적용되지 않았는데도 성능에는 큰 문제가 없었다.
- **ROC-AUC와 확률 보정**: 모든 예측 확률에 같은 값을 곱해도 순서가 그대로라 ROC-AUC는 변하지 않는다. 작성자가 주석으로 남긴 "very bad idea"가 그 이유를 보여 준다.
- **간단한 기본 모델의 가치**: 1위를 한 작성자도 공개한 노트북은 로그 변환, 오버샘플링, 랜덤 포레스트 정도의 짧은 파이프라인이었다. 복잡한 기법보다 먼저 데이터 특징(치우친 분포, 불균형)에 맞는 기본 처리를 하는 것이 중요하다는 점을 배웠다.

**아쉬운 점과 개선 방향**
- **표준화 결과를 저장하지 않음**: `scaler.fit_transform(X)`의 결과를 변수에 저장하지 않아 표준화가 적용되지 않았다. 랜덤 포레스트에는 영향이 적지만, 거리를 쓰는 ADASYN이나 다른 모델을 함께 쓴다면 문제가 된다.
- **한 번의 검증 세트**: 검증 데이터가 168개뿐이라 점수가 나누는 방식에 따라 크게 흔들린다. `StratifiedKFold`로 교차 검증을 하면 더 믿을 만한 점수를 얻을 수 있다.
- **시드를 고정하지 않음**: `train_test_split`, ADASYN, 랜덤 포레스트 모두 `random_state`가 없어서 실행할 때마다 결과가 달라진다.
- **하이퍼 파라미터 튜닝이 없음**: `n_estimators=1000`, `max_depth=40`을 고정값으로 썼다. 데이터가 작아서 깊이를 줄이거나 `min_samples_leaf`를 늘리면 과적합을 줄일 수 있을지 확인해 볼 만하다.
- **다른 모델과 비교하지 않음**: 랜덤 포레스트 하나만 사용했다. 앞에서 배운 XGBoost, LightGBM이나 로지스틱 회귀와 비교하고, 예측을 혼합하거나 스태킹하면 점수를 더 올릴 수 있을 것이다.

**내 프로젝트에 적용할 점**
- 분류 문제를 시작할 때 타깃 비율부터 확인하고, 불균형이 크면 정확도 대신 ROC-AUC, 재현율, 정밀도 같은 지표를 고른다.
- 오버샘플링은 반드시 학습·검증 분리 후 학습 데이터에만 적용한다.
- 전처리 함수의 결과를 변수에 제대로 저장했는지, 변환 후 평균이나 분포를 출력해서 확인하는 습관을 들인다.
- 시드를 고정해서 결과를 다시 확인할 수 있게 만든다.